# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and collect their `@id`s. All entities are referenced by their `@id` as per Croissant best practices.

In [ ]:
# Print an overview of all available record sets, fields, and columns with their @id.
print("Available record sets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"  RecordSet: @id = {rs.id}, name = {rs.name}")
    print("    Fields and columns in this RecordSet:")
    for field in rs.fields:
        print(f"      Field: @id = {field.id}, name = {field.name}, data_type = {field.data_type}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"        Column: @id = {col.id}, name = {col.name}, source = {col.source}")
    print()

## 3. Data Extraction

Load data from all record sets into DataFrames for further analysis. Below, use the record set `@id` (and field `@id`) you identified in the overview step above.

In [ ]:
# Collect the @id for all record sets
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_sets_ids:
    print(f"Available fields in record set '{record_sets_ids[0]}':")
    print(dataframes[record_sets_ids[0]].columns.tolist())
    display(dataframes[record_sets_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Use field `@id`s for reference.

In [ ]:
# Choose one record set for EDA (use the first, or select manually if known)
if record_sets_ids:
    selected_record_set_id = record_sets_ids[0]
    df = dataframes[selected_record_set_id]
    print(f"Analysis will use RecordSet @id: {selected_record_set_id}")

    # List all numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric columns detected: {numeric_cols}")

    # Select a numeric field for further processing
    if numeric_cols:
        numeric_field = numeric_cols[0]  # or set manually
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to pick a group field (categorical or string type)
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by '{group_field}':")
            print(grouped_df.head())
    else:
        print("No numeric columns available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Customize the code below with fields or groupings identified above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets_ids:
    df = dataframes[record_sets_ids[0]]
    # Example: Plot histogram of a numeric field
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_cols[0]].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_cols[0]}")
        plt.xlabel(numeric_cols[0])
        plt.show()

    # Example: Pairplot if multiple numeric columns
    if len(numeric_cols) > 1:
        sns.pairplot(df[numeric_cols].dropna())
        plt.suptitle("Pairplot of Numeric Fields")
        plt.show()

## 6. Conclusion

In this notebook, you loaded, explored, and performed simple analysis and visualizations on the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. The workflow can be adapted for further custom analyses, such as domain-specific feature engineering or modeling.

- All data handling referenced Croissant entities via `@id` for reproducibility and traceability.
- The EDA sections demonstrated filtering, normalization, grouping, and basic plotting.
- You can now extend this notebook to perform more advanced statistical or ML workflows.
